In [2]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '../data/raw/messy_mashup'
WORKING_BASE = '../data/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)




import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '../data/working/synthetic_mashups/train'
OUTPUT_DIR = '../data/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to ../data/working/features/train


In [18]:
import torch
import torch.nn as nn
import glob
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        return feature, label

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # TODO 1: CNN Backbone
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # TODO 2: RNN Component
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # TODO 3: Classifier
        self.fc = nn.Linear(128, num_classes)  # 64 * 2 because bidirectional


    def forward(self, x):

        # TODO 4: Pass through CNN
        x = self.cnn(x)

        # TODO 5: Bridge CNN → LSTM
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)        # (Batch, Time, Channels, Mels)
        x = x.reshape(b, t, c * f)       # (Batch, Time, 2048)

        # TODO 6: Pass through LSTM
        x, _ = self.lstm(x)

        # TODO 7: Global Max Pooling over time
        x, _ = torch.max(x, dim=1)

        # TODO 8: Final classifier
        logits = self.fc(x)

        return logits

### Question 1: You ran the generate_synthetic_dataset script with samples_per_genre=50. If the script executed successfully across all 10 target genres, exactly how many .wav files should now exist in your /kaggle/working/synthetic_mashups/train/ directory tree?

In [19]:
import glob
import os

# Path to generated mashups
mashup_dir = '../data/working/synthetic_mashups/train'

# Recursively find all .wav files
wav_files = glob.glob(os.path.join(mashup_dir, '**', '*.wav'), recursive=True)

# Print total count
print("Total .wav files generated:", len(wav_files))

Total .wav files generated: 500


### Question 2: Load any of your newly generated .wav files using torchaudio.load(). Given our configuration (target_sr=22050, duration=30), what is the exact tensor shape of the resulting waveform?
Provide in tuple format, example (x,y)

In [20]:
import torchaudio
import glob
import os

# Get any generated wav file
mashup_dir = '../data/working/synthetic_mashups/train'
wav_files = glob.glob(os.path.join(mashup_dir, '**', '*.wav'), recursive=True)

# Load the first file
waveform, sr = torchaudio.load(wav_files[0])

print("Sample Rate:", sr)
print("Waveform Shape:", waveform.shape)

Sample Rate: 22050
Waveform Shape: torch.Size([2, 661500])


### Question 3: Using the extract_and_save_features script with n_fft=2048, hop_length=512, and n_mels=128, you converted the waveforms into .pt files. 

If you load one of these pre-computed tensors using torch.load(), what is its exact dimension shape?

In [21]:
import torch
import glob
import os

# Path to saved feature tensors
features_dir = '../data/working/features/train'

# Get any .pt file
pt_files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)

# Load one tensor
mel_spec = torch.load(pt_files[0])

print("Mel Spectrogram Shape:", mel_spec.shape)

Mel Spectrogram Shape: torch.Size([2, 128, 1292])


/tmp/ipykernel_39774/3713411883.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mel_spec = torch.load(pt_files[0])


### Question 4: Inside the CRNN model's forward pass, the data moves from the CNN backbone to the LSTM. If your input batch has a size of 32, what is the exact shape of the tensor immediately after the second MaxPool2d(2) layer, right before the .permute() operation?

In [22]:
import torch

# Create model
model = CRNN()

# Dummy batch (Batch=32, Channels=1, Mel=128, Time=1292)
x = torch.randn(32, 1, 128, 1292)

# Pass through CNN only
cnn_out = model.cnn(x)

print("Shape after second MaxPool:", cnn_out.shape)

Shape after second MaxPool: torch.Size([32, 64, 32, 323])


### Question 5: Your model uses a Bidirectional LSTM with input_size=2048 and hidden_size=64. Write a small snippet using sum(p.numel() for p in model.lstm.parameters() if p.requires_grad) to calculate the exact number of trainable parameters in the LSTM layer alone. What is that integer value?

In [23]:
import torch

# Initialize the model
model = CRNN()

# Count trainable parameters in LSTM
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

print("Number of trainable parameters in LSTM:", lstm_params)

Number of trainable parameters in LSTM: 1082368


##### Now train the model using the following configuraiton:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10

In [24]:
from torch.utils.data import random_split

# Path to extracted features
features_dir = '../data/working/features/train'

# Load full dataset
full_dataset = PrecomputedFeatureDataset(features_dir)

# Train / Validation split (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 400
Validation samples: 100


In [25]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CRNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10


for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    train_loss = 0
    train_correct = 0
    total = 0

    for features, labels in train_loader:

        features = features.to(device)
        labels = labels.to(device)

        # FIX: convert stereo spectrograms to mono
        if features.shape[1] == 2:
            features = features.mean(dim=1, keepdim=True)

        optimizer.zero_grad()

        outputs = model(features)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = train_correct / total


    # -------- VALIDATION --------
    model.eval()

    val_loss = 0
    val_correct = 0
    total_val = 0

    with torch.no_grad():

        for features, labels in val_loader:

            features = features.to(device)
            labels = labels.to(device)

            # FIX: convert stereo spectrograms to mono
            if features.shape[1] == 2:
                features = features.mean(dim=1, keepdim=True)

            outputs = model(features)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            val_correct += (preds == labels).sum().item()
            total_val += labels.size(0)

    val_acc = val_correct / total_val


    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print("---------------------------------------------------")

/tmp/ipykernel_39774/593862310.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(file_path)


Epoch [1/10]
Train Loss: 28.6448 | Train Acc: 0.2675
Val Loss: 8.4156 | Val Acc: 0.2900
---------------------------------------------------
Epoch [2/10]
Train Loss: 24.8391 | Train Acc: 0.4325
Val Loss: 7.9596 | Val Acc: 0.3300
---------------------------------------------------
Epoch [3/10]
Train Loss: 23.1502 | Train Acc: 0.4775
Val Loss: 7.2957 | Val Acc: 0.4000
---------------------------------------------------
Epoch [4/10]
Train Loss: 21.2834 | Train Acc: 0.5350
Val Loss: 7.1858 | Val Acc: 0.4100
---------------------------------------------------
Epoch [5/10]
Train Loss: 19.9558 | Train Acc: 0.5725
Val Loss: 6.5758 | Val Acc: 0.5300
---------------------------------------------------
Epoch [6/10]
Train Loss: 18.5420 | Train Acc: 0.6250
Val Loss: 6.2957 | Val Acc: 0.5200
---------------------------------------------------
Epoch [7/10]
Train Loss: 17.5768 | Train Acc: 0.6575
Val Loss: 6.1086 | Val Acc: 0.5400
---------------------------------------------------
Epoch [8/10]
Train L